# 04 · 지표·분모·관찰 구간

목표: 누적 counter와 구간 비율을 구분하고 histogram을 읽는다. 약15분.

**이 파일의 핵심 질문:** 어떤 시간 구간·대상·실패 정의·분모로 지표를 계산했는가?

**Counter**는 누적 건수, **Gauge**는 현재 값, **Histogram**은 구간별 분포다. **le**는 less than or equal, 해당 경계 이하라는 뜻이다. **SLO**는 Service Level Objective, 서비스 수준 목표다. **PromQL**은 Prometheus Query Language, 시계열 조회 언어다.

**수업 중 먼저 실행할 단계**

- 1 관찰 전 스냅샷
- 2 기간 차이
- 3 텍스트 노출 형식
- 6 트래픽 없는 구간

파일 안의 변수 의존성이 있으므로 처음 실행할 때는 위에서부터 순서대로 진행한다. 핵심 단계에서는 출력을 읽고, 나머지 단계는 빠르게 실행한 뒤 복습 때 살펴본다.

## 시작 전 · 어디에서 무엇을 실행하나요?

1. VS Code에서 `llmops-16week` **폴더 전체**를 연다. 압축을 풀었다면 Notebook 파일만 따로 옮기지 않는다.
2. VS Code의 Python·Jupyter 확장을 설치하고 우측 위 **Select Kernel → Python Environments → .venv / Python 3.11**을 선택한다. 터미널의 가상환경과 Notebook 커널은 별도 선택이다.
3. 아래 **환경 확인 코드 셀부터** `Shift+Enter`로 한 셀씩 실행한다. 이 파일 안에서는 앞에서 만든 변수를 뒤에서 사용한다.
4. 각 단계의 **예상 결과**와 실제 출력을 대조한다. `assert`는 확인 조건이다. 출력이나 오류가 없어도 셀이 완료되면 조건을 통과한 것이다.
5. 셀을 건너뛰어 `NameError`가 나거나 변수를 여러 번 바꿨다면 **Restart Kernel → Run All**로 처음부터 실행한다. 다른 Notebook의 변수는 가져오지 않는다.

처음 환경을 만드는 경우에만 **VS Code 터미널**에서 실행한다. 아래 명령을 Python 코드 셀에 붙여넣지 않는다.

```bash
# macOS / Linux · 저장소 루트
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

Windows PowerShell은 `py -3.11 -m venv .venv`, `.venv\Scripts\Activate.ps1` 순서다. 활성화가 제한된 환경에서는 `.venv\Scripts\python.exe -m pip install -r requirements.txt`처럼 가상환경 Python을 직접 실행한다. 이미 준비된 `.venv`는 그대로 쓴다.

**공통 실습의 실행 방식:** `lab_client`가 메모리 안에 FastAPI 앱을 만들고 시작·종료를 관리한다. HTTPX의 ASGITransport가 같은 프로세스에서 요청을 전달하므로 `uvicorn`, Docker, Ollama, API 키가 필요하지 않다. ASGI는 *Asynchronous Server Gateway Interface*, Python 비동기 웹 앱과 서버 사이의 인터페이스다.

Notebook 앱과 브라우저의 `127.0.0.1:8010` 앱은 **서로 다른 실행**이다. Notebook에서 만든 요청 수가 브라우저·Prometheus에 합쳐지지 않는다. 실제 모델 대신 고정 문장과 대기를 쓰므로 외부 비용은 발생하지 않는다. 실행 시간·요청 ID는 매번 달라져도 정상이다.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import asyncio, json
from dataclasses import replace
from week10.lab.config import Settings
from week10.lab.experiments import lab_client, fixture_check, window_delta, compare_releases
assert sys.version_info[:2] == (3, 11), "Select the Python 3.11 .venv kernel in VS Code"
print("Python:", sys.version.split()[0])
print("실행 환경:", sys.executable)
print("저장소:", ROOT)
print("환경 확인 완료 · 이 Notebook은 별도 서버 없이 실행합니다.")

## 1. 관찰 전 스냅샷

상태 확인·UI·metrics 조회는 assist 분모에서 제외한다.

**예상 결과:** before.total=0, after.total=5. 정상2·서버 오류2·입력 오류1.

**결과 해석:** live·metrics 조회는 assist 요청 분모에 포함하지 않는다. 이 파일에서는 모든 완료 assist 요청을 분모로 쓴다.

In [ ]:
async with lab_client() as (app, client):
    await client.get("/health/live")
    await client.get("/metrics")
    before = app.state.telemetry.snapshot()
    for scenario in ["normal","normal","retrieval_error","provider_error"]:
        await client.post("/api/v1/assist", json={"message":"예시","scenario":scenario})
    await client.post("/api/v1/assist", json={"message":" "})
    after = app.state.telemetry.snapshot()
    exposition = (await client.get("/metrics")).text
assert before["total"] == 0 and after["total"] == 5

## 2. 기간 차이

5xx/전체 assist 요청으로 2/5를 계산한다. 422 포함 여부를 먼저 명시한다.

**예상 결과:** requests=5, 5xx=2, error_rate=0.4. 상태별 건수도 확인한다.

**결과 해석:** 이 실험의 오류 비율은2/5다. 입력 오류422를 제외하는 정책이라면 분모부터 다시 정의해야 한다.

In [ ]:
delta = window_delta(before, after)
assert delta["requests"] == 5 and delta["5xx"] == 2
assert delta["error_rate"] == 0.4
delta

## 3. 텍스트 노출 형식

counter·gauge·histogram이 어떤 줄인지 찾는다. 이 코드 셀은 수집 서버 없이 실행하며, 아래 선택 확장에서는 실제 Prometheus에 연결한다.

**예상 결과:** counter·gauge·histogram 텍스트. request_id·trace_id 라벨은 없다.

**결과 해석:** 현재 숫자를 노출하는 endpoint다. 아래 선택 실습에서 실제 Prometheus 서버가 별도로 이 텍스트를 수집한다.

In [ ]:
print(exposition)
assert "trace_id" not in exposition and "request_id" not in exposition

## 4. 누적 버킷

le=0.1은 0.1초 이하 전체다. 버킷을 단순 합산하면 중복된다.

**예상 결과:** 버킷 값이 순서대로 같거나 증가하고 +Inf 버킷은5.

**결과 해석:** 같은 요청이 여러 이하 경계에 포함된다. 버킷 값들을 단순 합산해 요청 수로 사용하지 않는다.

In [ ]:
buckets = [line for line in exposition.splitlines() if "_bucket" in line]
counts = [float(line.split()[-1]) for line in buckets]
assert counts == sorted(counts) and counts[-1] == 5
buckets

## 5. 성공 지연과 전체 지연

실패가 빨리 반환되면 전체 평균이 작아질 수 있다. logs는 최근 200건만이다.

**예상 결과:** 성공 평균과 전체 평균, 각각의 분모2·5.

**결과 해석:** 빠른 실패가 섞이면 전체 평균이 더 작아질 수 있다. 짧은 응답 시간이 내용 품질이나 성공의 증거가 되지는 않는다.

In [ ]:
success = [r["duration_ms"] for r in after["logs"] if r["status"] == 200]
all_ms = [r["duration_ms"] for r in after["logs"]]
print("성공 평균:",sum(success)/len(success), "전체 평균:",sum(all_ms)/len(all_ms))
print("분모:", len(success), len(all_ms))

## 6. 트래픽 없는 구간

요청0은 100% 성공이 아니라 관측 불충분이다.

**예상 결과:** 같은 snapshot 사이의 error_rate=None, 역순 차이는 ValueError 안내.

**결과 해석:** 요청0은 관측 부족이다. 재시작·다른 실행의 누적값을 그대로 빼서 비율을 계산하지 않는다.

In [ ]:
assert window_delta(after, after)["error_rate"] is None
try:
    window_delta(after, before)
except ValueError as e:
    print("재시작/역순 거부:", e)

## 7. 알림 판단

수업용 규칙. 운영 SLO·확정 배점이 아니다.

**예상 결과:** 관찰한 구간은 ‘원인 확인’, 요청0 구간은 ‘관측 부족’.

**결과 해석:** 예시 분기 규칙이다. 이 작은 표본의10%를 실제 서비스의 알림 기준으로 그대로 사용하지 않는다.

In [ ]:
def review_signal(d):
    return "관측 부족" if d["requests"] < 5 else "원인 확인" if d["error_rate"] > 0.1 else "관찰 유지"
print(review_signal(delta))
print(review_signal(window_delta(after,after)))

## 선택 확장 · 브라우저와 Prometheus에서 같은 지표 확인

공통 Notebook을 마친 뒤 진행한다. 여기부터는 **VS Code 터미널과 웹 브라우저**를 쓴다. Notebook 셀에서 Docker를 자동 시작하거나 패키지를 설치하지 않는다.

- **Docker:** 앱과 의존성을 이미지로 묶어 컨테이너로 실행.
- **Docker Compose:** 앱과 Prometheus 두 컨테이너의 설정·포트·연결을 함께 관리.
- **Prometheus:** 앱의 `/metrics`를5초마다 읽고 시간별 숫자를 저장·조회.
- **OpenTelemetry:** 앱의 계측과 데이터 전달을 위한 도구 모음. 이 저장소의 Trace·Span은 교육용 직접 구현이며 SDK·Collector는 포함하지 않는다. SDK는 Software Development Kit, 소프트웨어 개발 도구 모음이다.

### 1. 엔진 준비 · 최초 설치는 수업 전

Mac·Windows는 공식 Docker Desktop을 설치·실행하고 Linux는 Docker Engine과 Compose plugin을 준비한다. 이미 Colima를 쓰는 Mac은 기존 환경을 사용한다.

- [Mac 설치](https://docs.docker.com/desktop/setup/install/mac-install/)
- [Windows 설치](https://docs.docker.com/desktop/setup/install/windows-install/)
- [Linux 설치](https://docs.docker.com/engine/install/) · [Compose plugin](https://docs.docker.com/compose/install/linux/)

**터미널 A · 저장소 루트**에서 `docker info`의 Server 정보와 `docker compose version`을 확인한다. CLI는 Command Line Interface, 명령줄 인터페이스다. `docker --version`만으로 엔진의 실행 상태까지 확인한 것은 아니다. Colima context를 명시할 때는 각 명령의 `docker` 뒤에 `--context colima`를 붙인다.

### 2. 앱과 수집 서버 시작

기존8010포트의 Python 서버는 해당 터미널에서 Ctrl+C로 종료한다. 아래는 **터미널 A** 명령이다.

```bash
docker compose -p llmops-week10-demo -f week10/lab/observability/compose.yaml up -d --build
docker compose -p llmops-week10-demo -f week10/lab/observability/compose.yaml ps
```

최초 이미지를 내려받는 동안에는 시간이 걸린다. 앱이 healthy가 된 뒤 Prometheus가 시작된다. 브라우저에서 [수집 대상](http://127.0.0.1:9090/targets)을 열고 `week10-api`의 **UP**을 확인한다. 수집 주소는 컨테이너 서비스 이름인 `api:8010/metrics`다.

### 3. 요청30건 생성

**터미널 B · 저장소 루트**에서 기존 가상환경을 활성화한 뒤 실행한다. Mac/Linux는 `source .venv/bin/activate`, Windows는 `.venv\Scripts\Activate.ps1`이다.

```bash
python week10/lab/observability/seed_traffic.py --count 30
```

정상18·검색 오류6·시간 초과6건의 합성 요청이다. 새 앱에서 이 요청만 실행했다면 누적값도18·6·6이다. 기존 브라우저 요청이나 seed 재실행이 있으면 누적값이 증가한다. Notebook에서 만든5건은 이 서버에 합쳐지지 않는다.

### 4. 수집·누적값·변화율 구분

[Prometheus 조회 화면](http://127.0.0.1:9090/query)의 입력창에 **한 식씩** 넣는다. Python 코드 셀에 넣지 않는다.

```promql
up{job="week10-api"}
```

1이면 `/metrics` 수집 성공. 답변 내용의 정확성을 판정하는 값은 아니다.

```promql
lab_assist_requests_total{job="week10-api"}
```

Table에서 상태별 건수를 읽고 Graph로 바꾼다. 시간 범위를 최근5분으로 설정해 증가 구간을 본다. 두 오류의 건수가 같으면 선이 겹칠 수 있으므로 Table도 확인한다. 다음 수집까지 약5초 기다린다.

```promql
sum(rate(lab_assist_requests_total{job="week10-api"}[1m]))
```

최근1분의 요청/초다. 누적 건수30과 같은 값이 나오는 식이 아니다. 최소 두 번의 수집이 필요하고 트래픽 없는 구간은 빈 값·0이 나올 수 있다. 오류 비율의0/0은 NaN일 수 있으며 성공100%로 처리하지 않는다. 현재 실행 수 Gauge는 짧은 작업이 수집 사이에 끝나면0만 보일 수 있다.

### 5. 종료·문제 해결

```bash
docker compose -p llmops-week10-demo -f week10/lab/observability/compose.yaml down
```

자신이 만든 프로젝트만 정리한다. 실습 시계열은 tmpfs에 저장되어 종료 시 사라지고, 서버 재시작 뒤 counter도 초기화된다. 전역 정리 명령은 필요하지 않다.

| 증상 | 확인 |
|---|---|
| Cannot connect to Docker daemon | Docker Desktop·Colima 실행과 context 확인 |
| port already allocated | 본인이 실행한 Python 서버 종료 또는 QUICKSTART의 대체 포트 사용 |
| Targets DOWN | compose ps·API 준비 상태·`api:8010/metrics` 주소 확인 |
| 조회 결과 없음 | job 필터·실행한 서버·5초 수집 대기·Graph 시간 범위 확인 |
| 브라우저와 Notebook의 건수가 다름 | 각각 별도 앱이므로 정상. 같은 실행의 숫자끼리 비교 |

설정 전체·추가 조회식은 `week10/lab/observability/`와 `week10/QUICKSTART.md`에 있다. Docker 설치가 안 되어도 앞의 공통 실습은 완료할 수 있다.

## 직접 바꿔볼 실험 · 선택 복습

1. 사본의 첫 단계 시나리오 목록에 normal을1개 추가한다. 뒤의 assert도 새 기대값에 맞게 바꾸고 분모6·서버 오류2·오류 비율 약0.333을 확인한다.
2. 성공만 포함한 분모와 전체 완료 요청 분모의 차이를 문장으로 적는다.
3. 선택 Docker 실습에서 같은 지표의 누적값과 최근1분 rate를 비교한다.

한 번에 한 조건만 바꾸고 원래 코드와 비교한다. 기대값이 고정된 assert가 있다면 새 실험 조건에 맞는 값도 함께 설명한다.

## 프로젝트 적용 기록

**정리 질문:** 어떤 시간 구간·대상·실패 정의·분모로 지표를 계산했는가?

아래 Markdown 셀을 더블클릭해 자신의 결과를 적는다. 다른 파일 제출이나 화면 실행을 자동으로 수행하지 않는다.

| 항목 | 직접 기록 |
|---|---|
| 예상 | 실행 전 예상한 결과 |
| 관찰 | 상태 코드·건수·실패 단계 등 실제 결과 |
| 근거 | 실행한 단계 번호·변경한 조건·캡처 위치 |
| 판단 | 유지·수정·채택·보류 이유 |
| 미검증·다음 변경 | 아직 실행하지 않은 조건과 다음 비교 |

학생 정보나 원문 입력은 공개 저장소에 올리지 않는다. 실제 실행하지 않은 조건은 미측정으로 표시한다.

## 막혔을 때의 확인 순서

| 증상 | 확인·해결 |
|---|---|
| 커널 선택 창만 표시 | Python·Jupyter 확장 설치 후 `.venv`의 Python 선택 |
| `ModuleNotFoundError` | 커널 경로 확인 → 해당 `.venv`로 `python -m pip install -r requirements.txt` |
| 환경 확인에서 저장소를 찾지 못함 | 저장소 전체를 열고 원래 `week10/lab/notebooks` 위치에서 실행 |
| `NameError` | 앞 셀 실행 여부 확인 → Restart Kernel → Run All |
| `asyncio.run() cannot be called...` | Jupyter에서는 제공된 최상위 `await`·`async with`를 그대로 사용 |
| 셀에 출력이 없음 | `assert`만 있는 셀은 오류 없이 끝나면 통과. 왼쪽 실행 번호와 완료 표시 확인 |
| 예상한 422·502·503·504 | 입력 오류·검색 오류·준비 실패·시간 초과를 의도적으로 만든 단계인지 확인 |
| 예상과 다른 `AssertionError` | 바로 위 코드·출력 확인. 변경한 값을 복원하고 처음부터 다시 실행. assert를 지워 통과시키지 않음 |
| 실행 시간·ID가 예시와 다름 | 고정값 비교 대신 상태·부모/자식 관계·분모·검증 결과를 비교 |

실습에서 바꾼 내용은 자신의 사본에 기록한다. 공개 저장소에는 학생 정보·원문 질문·실제 키를 올리지 않는다.